In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().absolute().parent))


In [2]:
from src.data_models.caravanify import Caravanify, CaravanifyConfig


In [3]:
import pandas as pd 
import geopandas as gpd

---

In [4]:
kyrg_path = "/Users/cooper/Desktop/CAMELS-CH/data/CA_quantile_mapped/HRU00003_forcing_2000-2023.csv"
tajik_path = "/Users/cooper/Desktop/CAMELS-CH/data/CA_quantile_mapped/hydro_TJK.csv"

# Read the CSV file into a DataFrame
df_kyrg = pd.read_csv(kyrg_path)
df_tajik = pd.read_csv(tajik_path)

# Turn date to datetime
df_kyrg['date'] = pd.to_datetime(df_kyrg['date'])
df_tajik['date'] = pd.to_datetime(df_tajik['date'])

# Filter for 2000 onwards
df_kyrg = df_kyrg[df_kyrg['date'] >= '2000-01-01']
df_tajik = df_tajik[df_tajik['date'] >= '2000-01-01']

# Rename code to gauge_id, P to total_precipitation_sum_qm, T to temperature_2m_mean_qn
df_kyrg = df_kyrg.rename(columns={'code': 'gauge_id', 'P': 'total_precipitation_sum_qm', 'T': 'temperature_2m_mean_qn'})
df_tajik = df_tajik.rename(columns={'code': 'gauge_id', 'P': 'total_precipitation_sum_qm', 'T': 'temperature_2m_mean_qn'})

df_kyrg = df_kyrg[['date', 'gauge_id', 'total_precipitation_sum_qm', 'temperature_2m_mean_qn']]
df_tajik = df_tajik[['date', 'gauge_id', 'total_precipitation_sum_qm', 'temperature_2m_mean_qn']]

# Add CA_ prefix to gauge_id
df_kyrg['gauge_id'] = 'CA_QM_' + df_kyrg['gauge_id'].astype(str)
df_tajik['gauge_id'] = 'CA_QM_' + df_tajik['gauge_id'].astype(str)

# Drop gauge_ids CA_16936 and CA_17084
df_kyrg = df_kyrg[~df_kyrg['gauge_id'].isin(['CA_16936', 'CA_17084'])]
df_tajik = df_tajik[~df_tajik['gauge_id'].isin(['CA_16936', 'CA_17084'])]

## The basins with ids CA_16936 (Kyrg) and CA_17084 (Tajik) are not present in CARAVANIFY

In [5]:
config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)


caravan = Caravanify(config)
ids_for_training = caravan.get_all_gauge_ids()

caravan.load_stations(ids_for_training)
ts_data = caravan.get_time_series()[["gauge_id", "date", "streamflow"]]

In [6]:
# Add a streamflow column to the Kyrgyzstan and Tajikistan dataframes by merging with the timeseries data
df_kyrg = df_kyrg.merge(ts_data, on=["gauge_id", "date"], how="left")
df_tajik = df_tajik.merge(ts_data, on=["gauge_id", "date"], how="left")

# Merge the two dataframes
df_merged = pd.concat([df_kyrg, df_tajik], ignore_index=True)

In [7]:
df_merged

,date,gauge_id,total_precipitation_sum_qm,temperature_2m_mean_qn,streamflow
0,2000-01-01,CA_QM_16070,0.271568,-13.984251,NaN
1,2000-01-01,CA_QM_16160,0.000000,-10.369381,NaN
2,2000-01-01,CA_QM_15049,0.033652,-10.882885,NaN
3,2000-01-01,CA_QM_15044,0.065961,-12.002902,NaN
4,2000-01-01,CA_QM_16096,0.138954,-9.884873,NaN
...,...,...,...,...,...
694098,2023-12-27,CA_QM_17453,10.492144,-14.560862,NaN
694099,2023-12-28,CA_QM_17453,0.210808,-18.548800,NaN
694100,2023-12-29,CA_QM_17453,0.511145,-16.299217,NaN
694101,2023-12-30,CA_QM_17453,0.750811,-16.737628,NaN


In [8]:
new_folder = "/Users/cooper/Desktop/CAMELS-CH/data/CA_quantile_mapped/time_series_QM"

# Create the new folder if it doesn't exist
Path(new_folder).mkdir(parents=True, exist_ok=True)

In [9]:
# For each unique gauge_id, create a new CSV file and call it {gauge_id}.csv
for gauge_id in df_merged['gauge_id'].unique():
    # Filter the dataframe for the current gauge_id
    df_gauge = df_merged[df_merged['gauge_id'] == gauge_id]
    
    # Save the dataframe to a CSV file
    df_gauge.to_csv(f"{new_folder}/{gauge_id}.csv", index=False)


---

In [12]:
from concurrent.futures import ThreadPoolExecutor

time_series = {}

ts_dir = Path(new_folder)
file_paths = []
for gauge_id in df_merged["gauge_id"].unique():
    fp = ts_dir / f"{gauge_id}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Timeseries file {fp} not found")
    file_paths.append(fp)


def read_single(fp: Path) -> pd.DataFrame:
    # Always use pyarrow engine for faster parsing
    df = pd.read_csv(fp, parse_dates=["date"], engine="pyarrow")
    df["gauge_id"] = fp.stem
    return df


with ThreadPoolExecutor() as executor:
    dfs = list(executor.map(read_single, file_paths))

for df in dfs:
    time_series[df["gauge_id"].iloc[0]] = df


df = pd.concat(time_series.values(), ignore_index=True)
df[["gauge_id", "date"] + [c for c in df.columns if c not in ("gauge_id", "date")]]

,gauge_id,date,total_precipitation_sum_qm,temperature_2m_mean_qn,streamflow
0,CA_QM_16070,2000-01-01,0.271568,-13.984251,NaN
1,CA_QM_16070,2000-01-02,1.508941,-12.960913,NaN
2,CA_QM_16070,2000-01-03,4.557064,-11.794987,NaN
3,CA_QM_16070,2000-01-04,0.794701,-15.243411,NaN
4,CA_QM_16070,2000-01-05,0.020802,-17.848515,NaN
...,...,...,...,...,...
694098,CA_QM_17453,2023-12-27,10.492144,-14.560862,NaN
694099,CA_QM_17453,2023-12-28,0.210808,-18.548800,NaN
694100,CA_QM_17453,2023-12-29,0.511145,-16.299217,NaN
694101,CA_QM_17453,2023-12-30,0.750811,-16.737628,NaN


In [13]:
df

,date,gauge_id,total_precipitation_sum_qm,temperature_2m_mean_qn,streamflow
0,2000-01-01,CA_QM_16070,0.271568,-13.984251,NaN
1,2000-01-02,CA_QM_16070,1.508941,-12.960913,NaN
2,2000-01-03,CA_QM_16070,4.557064,-11.794987,NaN
3,2000-01-04,CA_QM_16070,0.794701,-15.243411,NaN
4,2000-01-05,CA_QM_16070,0.020802,-17.848515,NaN
...,...,...,...,...,...
694098,2023-12-27,CA_QM_17453,10.492144,-14.560862,NaN
694099,2023-12-28,CA_QM_17453,0.210808,-18.548800,NaN
694100,2023-12-29,CA_QM_17453,0.511145,-16.299217,NaN
694101,2023-12-30,CA_QM_17453,0.750811,-16.737628,NaN
